In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [ ]:
visual1 = widgets.Output()
visual2 = widgets.Output()
visual3 = widgets.Output()
visual4 = widgets.Output()
visual5 = widgets.Output()
visual6 = widgets.Output()


title = widgets.HTML("<h1 style='text-align: left;'>Colon Cancer and It's Risk Factors</h1>")
subtitle = widgets.HTML("<h3 style='text-align: left;'>An Analytical Overview of Key Risk Factors and Trends</h3>")

with visual1:

    display(widgets.HTML("<h2>Visualization 1: Colon Cancer Incidence by Location</h2><p></p>"))
    
    cancer_data = pd.read_csv("colon_cancer_incidence.csv")


    # Prepare data for Plotly visualization
    cancer_data.rename(columns=lambda x: x.replace('N', '').replace('_','-') if x.startswith('N') else x, inplace=True)
    locations = sorted(cancer_data['Location'].unique())
    age_groups = [col for col in cancer_data.columns if col[0].isdigit()]
    
    
    # Create a figure with dropdown functionality
    fig1 = go.Figure()
    
    # Add traces for each location with custom hover template
    for location in locations:
        location_data = cancer_data[cancer_data['Location'] == location]
        incidence_by_age = location_data[age_groups].sum().values
    
        fig1.add_trace(go.Bar(
            x=age_groups,
            y=incidence_by_age,
            name=location,
            visible=(location == locations[0]),  # Only the first location is visible initially
            hovertemplate=
            "<b>Location</b>: " + location + "<br>" +
            "<b>Age Group</b>: %{x}<br>" +
            "<b>Number of Cases</b>: %{y}<br>" +
            "<extra></extra>"  # This removes the secondary box in the hover tooltip
        ))
    
    # Update layout with dropdown menu
    fig1.update_layout(
        updatemenus=[
            {
                "buttons": [
                    {
                        "label": loc,
                        "method": "update",
                        "args": [{"visible": [loc == l for l in locations]}]
                    } for loc in locations
                ],
                "direction": "down",
                "showactive": True,
                "x": 0.5,  # Center the dropdown
                "xanchor": "center",  # Align the center of the dropdown to the x position
                "y": 1.15,  # Position above the plot
                "yanchor": "top",  # Align the top of the dropdown
                "pad": {"l": 0, "r": 0, "t": 0, "b": 0},  # Minimize padding
            }
        ],
        title="Cancer Incidence by Age Group and Location",
        xaxis_title="Age Group",
        yaxis_title="Number of Cases",
        barmode="group",
        hoverlabel=dict(
            bgcolor="white",
            font_size=12,
            font_family="Rockwell"
        )
    )
    fig1.show()
with visual2:
    display(widgets.HTML("<h2>Visualization 2: Colon Cancer vs Tobacco</h2><p></p>"))

    long_data = pd.read_csv("long_data.csv")
    fig2 = px.scatter(
        long_data,
        x="Numeric",  # Tobacco use
        y="Cancer_Incidence",
        color="Gender",
        hover_data=["Country", "Age_Group"],
        title="Tobacco Use vs. Colon Cancer Incidence",
        labels={'Numeric':'Tobacco Usage (% of population)', 'Cancer_Incidence': 'Total Cancer Incidents'}
)

    fig2.show()

with visual3:
    display(widgets.HTML("<h2>Visualization 3: Colon Cancer by Continent</h2><p></p>"))
 
    melted = pd.read_csv("melted.csv")
    fig3 = go.Figure(data=go.Heatmap(
    x=melted['Age_Group'],
    y=melted['Continent'] + " - " + melted['Gender'],
    z=melted['Cancer_Incidence'],
    colorscale='Blues',
    colorbar=dict(title="Cancer Incidence"),
    hovertemplate="<b>Continent:</b> %{y}<br><b>Age Group:</b> %{x}<br>" +
                  "<b>Cancer Incidence:</b> %{z}<br>" +
                  "<b>Average Tobacco Usage:</b> %{customdata}<extra></extra>",
    customdata=melted['Numeric']  
    ))
    
    # Set title and labels
    fig3.update_layout(
        title="Cancer Incidence by Age Group and Continent",
        xaxis_title="Age Group",
        yaxis_title="Continent and Gender",
        yaxis=dict(tickmode="linear")
    )
    
    fig3.show()

with visual4:
    display(widgets.HTML("<h2>Visualization 4: Salt Intake Map</h2><p></p>"))

    salt2 = pd.read_csv('./Salt_Intake2.csv')
    colon2 = pd.read_csv('./colon_cancer_incidence2.csv')
    # Filter for relevant rows in salt intake dataset
    salt_filtered = salt2[salt2['Indicator Name'] == 'Mean population salt intake, adults aged 25+']
    salt_filtered = salt_filtered[['Country Name', 'Numeric']].rename(columns={'Country Name': 'Country', 'Numeric': 'Salt_Intake'})
    salt_filtered['Salt_Intake'] = pd.to_numeric(salt_filtered['Salt_Intake'], errors='coerce')
    
    
    # Clean and aggregate the colon cancer data
    # Extract country names by splitting at commas and taking the first part
    colon2['Country'] = colon2['Location'].str.split(',').str[0]
    # Sum incidence rates for each country
    colon_filtered = colon2.groupby('Country', as_index=False)['TOTAL'].sum().rename(columns={'TOTAL': 'Colon_Cancer_Rate'})
    colon_filtered = colon_filtered[colon_filtered['Country'] != "United States: Black"]
    colon_filtered = colon_filtered[colon_filtered['Country'] != "United States: White"]
    # Merge datasets on the 'Country' column
    combined_df = pd.merge(salt_filtered, colon_filtered, on='Country')
    
    # Plot for salt intake
    fig4 = px.choropleth(
        salt_filtered,
        locations="Country",
        locationmode="country names",
        color="Salt_Intake",
        hover_name="Country",
        color_continuous_scale="YlOrBr",
        title="Mean Population Salt Intake (g/day))"
    )
    fig4.update_layout(
    width=900,  
    height=800   
)
    fig4.show()
with visual5:
    display(widgets.HTML("<h2>Visualization 5: Colon Cancer Map</h2><p></p>"))

    fig5 = px.choropleth(
    colon_filtered,
    locations="Country",
    locationmode="country names",
    color="Colon_Cancer_Rate",
    hover_name="Country",
    color_continuous_scale="YlOrRd",
    title="Colon Cancer Rate by Country"
)
    fig5.update_layout(
    width=900,  
    height=800   
)
    fig5.show()

with visual6:
    display(widgets.HTML("<h2>Visualization 6: More Interesting Factors</h2><p></p>"))
    
    merged_data = pd.read_csv("merged_data.csv")
    correlation_matrix = merged_data[['Cancer Incidence', 'Alcohol_Consumption', 'Obesity_Prevalence', 'Physical_Inactivity']].corr()
fig6 = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Cancer Incidence and More Risk Factors"
)
fig6.show()

# Layout setup
row1 = widgets.VBox([title, subtitle, visual1])
row2 = widgets.HBox([visual2])
row3 = widgets.HBox([visual3])
row4 = widgets.HBox([visual4, visual5])
row5 = widgets.VBox([visual6])

# Combine all rows into a vertical layout
dashboard = widgets.VBox([row1, row2, row3, row4, row5])

# Display the dashboard
display(dashboard)

In [7]:
from pathlib import Path
Path("docs").mkdir(exist_ok=True)

figures = {
    "incidence_by_location": fig1,
    "tobacco_vs_incidence": fig2,
    "incidence_by_continent": fig3,
    "salt_intake_map": fig4,
    "colon_cancer_map": fig5,
    "other_risk_factors": fig6,
}

for name, f in figures.items():
    f.write_html(f"docs/{name}.html", include_plotlyjs="cdn")

In [8]:
titles = {
    "incidence_by_location": "Visualization 1: Colon Cancer Incidence by Location",
    "tobacco_vs_incidence": "Visualization 2: Colon Cancer vs Tobacco",
    "incidence_by_continent": "Visualization 3: Colon Cancer by Continent",
    "salt_intake_map": "Visualization 4: Salt Intake Map",
    "colon_cancer_map": "Visualization 5: Colon Cancer Map",
    "other_risk_factors": "Visualization 6: More Interesting Factors",
}

sections = []
for i, (name, f) in enumerate(figures.items()):
    chart = f.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False)
    sections.append(f"<h2>{titles[name]}</h2>\n{chart}")

page = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Colon Cancer and Its Risk Factors</title>
  <style>
    body {{ font-family: system-ui, sans-serif; max-width: 1000px; margin: 2rem auto; padding: 0 1rem; }}
  </style>
</head>
<body>
  <h1>Colon Cancer and Its Risk Factors</h1>
  <h3>An Analytical Overview of Key Risk Factors and Trends</h3>
  {''.join(sections)}
</body>
</html>"""

Path("docs/index.html").write_text(page, encoding="utf-8")

2823589

In [9]:
for p in sorted(Path("docs").glob("*.html")):
    print(f"{p.name}: {p.stat().st_size / 1024:.0f} KB")

age_group_incidence.html: 911 KB
colon_cancer_map.html: 10 KB
incidence_by_continent.html: 18 KB
incidence_by_location.html: 2393 KB
index.html: 2757 KB
other_risk_factors.html: 9 KB
salt_intake_map.html: 29 KB
tobacco_vs_incidence.html: 299 KB
